In [ ]:
# This is the implementation of the radiative-transfer model
# calibrated in https://ieeexplore.ieee.org/document/11611206
# to simulate the emssivity of the Moon

In [ ]:
# Allows the modification of the files imported without having to restart kernel or re-import them
%load_ext autoreload
%autoreload 2
# to have interactive plots : pip install ipympl
%matplotlib ipympl

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
from astropy import units as u
from datetime import datetime
from astropy.time import Time
from astropy.coordinates import EarthLocation, AltAz, get_body
import pipeline_moon_functions as pmf

In [ ]:
nu = 150e9 # Hz
phi = 10 # deg
dist_MS = pmf.AU_meters # m
Tb_test = pmf.get_disc_integrated_Moon_TB_interp(nu, phi, dist_MS)

In [ ]:
Tb_test

In [ ]:
ObsSite = pmf.LaPuna_QUBIC
Obs_Site = EarthLocation(lat=ObsSite["lat"], lon=ObsSite["lon"], height=ObsSite["height"])
longitude = float(Obs_Site.lon/u.deg)
latitude = float(Obs_Site.lat/u.deg)
# times at which we want to know the position of the Moon
tinit = 1785020764 # random test with 25/07/2026 23:06:04
tt = np.linspace(0, 1000, 1001)
date_obs = str(datetime.utcfromtimestamp(tinit))
print('Observations started at: {} UTC (old)'.format(date_obs))

alltimes = Time(date_obs) + tt*u.second

### Moon
moon_observed = get_body('Moon', alltimes, Obs_Site)

# Computing the mean angular size of the Moon
moon_dist = moon_observed.distance.value * pmf.AU_meters # m
moon_radius = 1737.4 * 1e3 # m
moon_angsize = np.mean(np.arctan(2 * moon_radius / moon_dist)) # radians
moon_solid_angle = np.pi * moon_radius**2 / moon_dist**2 # steradian

In [ ]:
print(0.984 * pmf.AU_meters, 1.016 * pmf.AU_meters)
print(0.033 * 1e11)
print(pmf.AU_meters/(0.033 * 1e11))

In [ ]:
phi = np.linspace(-100, 50, 1000)
dist_MS = np.array([0.984, 1, 1.016]) * pmf.AU_meters
ls = ["--", "-", "-."]
colors = ["red", "green", "blue"]
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
for i_freq in range(len(pmf.nus_calib) - 1):
    Tbs_dist = []
    for i_dist in range(len(dist_MS)):
        Tbs = pmf.get_disc_integrated_Moon_TB_calib(i_freq, phi, dist_MS[i_dist])
        Tbs_dist.append(Tbs)
        axs[0].plot(phi, Tbs, c=colors[i_freq], ls=ls[i_dist], label="{} GHz".format(pmf.nus_calib[i_freq]))
    axs[1].plot(phi, Tbs_dist[0] - Tbs_dist[1], c=colors[i_freq], label="{} GHz".format(pmf.nus_calib[i_freq]))
plt.legend()
plt.show()

In [ ]:
phi = 0
dist_MS = np.linspace(0.984, 1.016, 10) * pmf.AU_meters
colors = ["red", "green", "blue"]
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.set_title("{} GHz, delta Tb [K] vs dist_MS [m]".format(pmf.nus_calib[i_freq]))
Tbs_dist = []
i_freq = 0
ref = pmf.get_disc_integrated_Moon_TB_calib(i_freq, phi, 1*pmf.AU_meters)
for i_dist in range(len(dist_MS)):
    Tbs = pmf.get_disc_integrated_Moon_TB_calib(i_freq, phi, dist_MS[i_dist])
    Tbs_dist.append(Tbs)
Tbs_dist = np.array(Tbs_dist)
ax.plot(dist_MS, Tbs_dist - ref, c=colors[i_freq])
plt.legend()
plt.show()

In [ ]:
phis = np.linspace(-90, 90, 5)
dist_MS = 1 * pmf.AU_meters
colors = ["red", "green", "blue"]
fig1, ax1 = plt.subplots(1, 1, figsize=(10, 5))
fig2, ax2 = plt.subplots(1, 1, figsize=(10, 5))
i_freqs = np.arange(4)
for i_phi, phi in enumerate(phis):
    Tbs_freq = []
    I_nu = []
    for i_freq in i_freqs:
        Tbs = pmf.get_disc_integrated_Moon_TB_calib(i_freq, phi, dist_MS)
        Tbs_freq.append(Tbs)
        I_nu.append(pmf.Tb2I(Tbs, pmf.nus_calib[i_freq]))
    # Tbs_freq = np.array(Tbs_freq)
    ax1.plot(pmf.nus_calib, Tbs_freq, label="Lunar phase angle = {}°".format(phi))
    ax2.plot(pmf.nus_calib, I_nu, label="Lunar phase angle = {}°".format(phi))
ax1.set_title("Tb [K] vs nu [GHz]")
ax2.set_title("I_nu [W m-2 Hz-1 sr-1] vs nu [GHz]")
fig1.legend()
fig2.legend()
plt.show()

In [ ]:
phis = np.linspace(-90, 90, 5)
dist_MS = 1 * pmf.AU_meters
colors = ["red", "green", "blue"]
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.set_title("{} GHz".format(pmf.nus_calib[i_freq]))
nus = np.linspace(np.min(pmf.nus_calib) + 1, np.max(pmf.nus_calib) - 1, 20) * 1e9 # Hz
# i_freqs = np.arange(4)
for i_phi, phi in enumerate(phis):
    Tbs_freq = pmf.get_disc_integrated_Moon_TB_interp(nus, phi, dist_MS)
    ax.plot(nus, Tbs_freq, label="Lunar phase angle = {}°".format(phi))
plt.legend()
plt.show()

In [ ]:
nside = 512
az_moon = np.float_(90) # deg
el_moon = np.float_(50) # deg
moon_pos = pmf.spherical2cartesian(rho=1, theta=az_moon, phi=el_moon, coord="horizontal")
Tb = 250 # K
nu_interval = np.array([130, 170]) * 1e9 # Hz
map_moon, power_pixel = pmf.get_moon_map(nu_interval, nside, moon_pos, moon_angsize, phi, dist_MS)

In [ ]:
hp.mollview(map_moon, xsize=1000, cmap="grey") # xsize=10000 with nside = 1024 to really see a circle
plt.show()

In [ ]:
hp.gnomview(map_moon, rot=[az_moon, el_moon], reso=0.1, xsize=1000, cmap="grey")
plt.show()